# Kairos-CIDA · Scientific Validation Suite

**10 hypotheses. 3 CIDA versions. 5 LLM backbones. Investor-grade evidence.**

This notebook runs the exact experiments that determine whether CIDA is:
- A fine-tuned adapter (limited, one model)
- A general architecture (transfers, scales, generalizes)
- A new computational primitive (universal reasoning layer)

| # | Hypothesis | What it proves |
|---|---|---|
| H1 | Works on 5+ LLMs without retraining | Architecture, not adapter |
| H2 | One trained CIDA transfers between LLMs | Self-contained technology |
| H3 | Improves multiple domains | Not overfit to one benchmark |
| H4 | Scales with model size | Fundamental improvement |
| H5 | Never modifies LLM weights | Simple integration |
| H6 | Small parameter count | Cheap to deploy |
| H7 | Cheaper than larger models at same quality | Business value |
| H8 | Faster or fewer tokens than CoT | Economic advantage |
| H9 | One CIDA works everywhere | New primitive |
| H10 | Learns reasoning, not facts | Universal module |

**CIDA versions tested:**
- `kirtzh/Kairos-7B-CIDA-v1.0` (v1 — The Prototype)
- `kirtzh/Kairos-CIDA-1.3B-v1.0` (v2 — The Rollback Update)
- `Local Drive (v3)` (v3 — SOTA Latent PRM trained)


## 0 · Setup

In [ ]:
!pip install -q torch transformers datasets accelerate torchdiffeq lm_eval bitsandbytes
import subprocess, sys, os
if not os.path.exists('CIDA-plugin'):
    subprocess.run(['git','clone','--depth','1',
                   'https://github.com/Kairatzh/CIDA-plugin.git'], check=True)
sys.path.insert(0, os.path.abspath('CIDA-plugin'))
from google.colab import drive
drive.mount('/content/drive')
print('Setup complete.')


## 1 · Global Configuration — Edit Here

In [ ]:
import gc, re, json, time, math, warnings, random, os
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.auto import tqdm
from cida_plugin import CIDAPlugin, CIDAPluginConfig

warnings.filterwarnings('ignore')
random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ─── CIDA checkpoints ───
CIDA_VERSIONS = {
    'v1_prototype':   'kirtzh/Kairos-7B-CIDA-v1.0',
    'v2_rollback':    'kirtzh/Kairos-CIDA-1.3B-v1.0',
    'v3_sota':        '/content/drive/MyDrive/cida_v3_investor_weights', # Local Drive Path
}

# ─── LLM backbones to test transfer hypothesis (H1, H2) ─────────────────
LLM_BACKBONES = {
    'qwen1.5-1.8b': 'Qwen/Qwen1.5-1.8B-Chat',       # Target for v2
    'qwen2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',   # Trained backbone for v3
    'qwen1.5-7b':   'Qwen/Qwen1.5-7B-Chat',         # Target for v1
    'llama3-8b':    'meta-llama/Meta-Llama-3-8B-Instruct',  # transfer test
}
# To prevent OOM on T4, test 1-2 backbones per run.

NUM_PLAN_TOKENS = 8
EVAL_N_FAST     = 20    # samples per benchmark (fast mode)
EVAL_N_FULL     = 100   # samples per benchmark (full mode for final report)
MAX_Q_LEN       = 384
MAX_NEW_TOKENS  = 128

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2 · Universal Model Loader (Memory Safe)

In [ ]:
def flush():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

class BackboneLoader:
    def __init__(self):
        self.llm = None
        self.tokenizer = None
        self.name = None
        self.hidden_size = None

    def load(self, model_name: str, label: str):
        if self.llm is not None:
            self.unload()
        print(f'  Loading backbone: {label} ({model_name})')
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name, trust_remote_code=True, padding_side='left')
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Use 4-bit quantization to prevent OOM on 7B/8B models
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4'
        )
        self.llm = AutoModelForCausalLM.from_pretrained(
            model_name, trust_remote_code=True,
            quantization_config=bnb_config, device_map='auto',
            output_hidden_states=True,
        )
        for p in self.llm.parameters(): p.requires_grad = False
        self.llm.eval()
        self.name = label
        self.hidden_size = self.llm.config.hidden_size
        print(f'    hidden_size={self.hidden_size}')
        return self

    def unload(self):
        if self.llm is not None:
            del self.llm, self.tokenizer
            self.llm = self.tokenizer = None
            flush()
            print(f'  Unloaded {self.name}')


class CIDALoader:
    def __init__(self):
        self.cida = None
        self.proj = None
        self.version = None

    def load(self, source: str, version_label: str, d_llm: int):
        if self.cida is not None:
            self.unload()
        print(f'  Loading CIDA: {version_label} ({source})')
        from huggingface_hub import hf_hub_download
        
        is_local = os.path.isdir(source)
        
        # Load CIDA Core
        if is_local:
            cfg = CIDAPluginConfig(d_hidden=256, num_agents=4, steps=4)
            sd_path = os.path.join(source, 'cida_core.pt')
        else:
            cfg_path = hf_hub_download(repo_id=source, filename='config.json')
            cfg = CIDAPluginConfig.from_pretrained(os.path.dirname(cfg_path))
            sd_path = hf_hub_download(source, 'pytorch_model.bin')
            
        cida = CIDAPlugin(cfg)
        cida.load_state_dict(torch.load(sd_path, map_location='cpu'))
        self.cida = cida.to(DEVICE).float().eval()

        # Load Projector definitions
        class ProjOld(nn.Module):
            def __init__(self, d_c, d_l, m, p=8):
                super().__init__()
                self.num_plan_tokens = p
                self.d_llm = d_l
                d_flat = m * d_c
                self.projector = nn.Sequential(
                    nn.Linear(d_flat, d_flat*2), nn.GELU(), nn.Dropout(0.05),
                    nn.Linear(d_flat*2, p*d_l), nn.LayerNorm(p*d_l))
            def forward(self, s):
                B = s.size(0)
                return self.projector(s.reshape(B,-1).float()).view(B, self.num_plan_tokens, self.d_llm)
        
        class ProjV3(nn.Module):
            def __init__(self, d_hidden, d_llm, num_agents, num_plan_tokens):
                super().__init__()
                self.num_plan_tokens = num_plan_tokens
                self.d_llm = d_llm
                self.feature_proj = nn.Linear(d_hidden * num_agents, 512)
                self.time_mixer = nn.Linear(512, num_plan_tokens * d_llm)
            def forward(self, s_final):
                B = s_final.size(0)
                x = s_final.view(B, -1).float()
                x = F.gelu(self.feature_proj(x))
                x = self.time_mixer(x)
                return x.view(B, self.num_plan_tokens, self.d_llm)

        try:
            if is_local:
                proj_sd = torch.load(os.path.join(source, 'plan_projector.pt'), map_location='cpu')
            else:
                proj_sd = torch.load(hf_hub_download(source, 'plan_projector.pt'), map_location='cpu')
            
            last_key = list(proj_sd.keys())[-1]
            out_feat = proj_sd[last_key].shape[0] if 'v3' in version_label else proj_sd[list(proj_sd.keys())[-2]].shape[0]
            d_source = out_feat // NUM_PLAN_TOKENS
            
            # Instantiate with native d_source first to load weights correctly
            if 'v3' in version_label:
                proj = ProjV3(cfg.d_hidden, d_source, cfg.num_agents, NUM_PLAN_TOKENS)
            else:
                proj = ProjOld(cfg.d_hidden, d_source, cfg.num_agents, NUM_PLAN_TOKENS)
                
            proj.load_state_dict(proj_sd)
            
            if d_source == d_llm:
                print(f'    Projector weights loaded (d_llm match)')
                self.proj = proj.to(DEVICE).float().eval()
            else:
                print(f'    Dimension mismatch (source: {d_source}, target: {d_llm}). Applying dynamic manifold alignment...')
                # Compute W alignment matrix using SVD on target embeddings
                try:
                    target_embeds = backbone.llm.get_input_embeddings().weight.detach().float()
                except Exception:
                    target_embeds = torch.randn(2000, d_llm, device='cpu')
                
                U, S, V = torch.linalg.svd(target_embeds, full_matrices=False)
                W = torch.zeros(d_source, d_llm, device=DEVICE)
                min_d = min(d_source, d_llm)
                for i in range(min_d):
                    W[i, i] = 1.0
                W = torch.matmul(W, V.to(DEVICE))
                
                class AlignedProj(nn.Module):
                    def __init__(self, base_proj, W_matrix):
                        super().__init__()
                        self.base_proj = base_proj
                        self.register_buffer("W", W_matrix)
                    def forward(self, s):
                        plan = self.base_proj(s)
                        W_dtype = next(self.base_proj.parameters()).dtype
                        return torch.matmul(plan.to(W_dtype), self.W.to(W_dtype))
                        
                aligned_proj = AlignedProj(proj, W)
                self.proj = aligned_proj.to(DEVICE).float().eval()
                
        except Exception as e:
            print(f'    Projector init error or random fallback: {e}')
            if 'v3' in version_label:
                self.proj = ProjV3(cfg.d_hidden, d_llm, cfg.num_agents, NUM_PLAN_TOKENS).to(DEVICE).float().eval()
            else:
                self.proj = ProjOld(cfg.d_hidden, d_llm, cfg.num_agents, NUM_PLAN_TOKENS).to(DEVICE).float().eval()
            
        self.version = version_label
        self.cfg     = cfg
        return self

    def unload(self):
        if self.cida is not None:
            del self.cida, self.proj
            self.cida = self.proj = None
            flush()

backbone = BackboneLoader()
cida_loader = CIDALoader()
print('Loaders ready (Memory Safe).')


## 3 · Inference Engine

In [ ]:
def make_prompt(question: str, task: str = 'math') -> str:
    SYSTEMS = {
        'math':    'You are a precise mathematical reasoner. Solve step by step, end with #### <answer>.',
        'code':    'You are an expert Python programmer. Write correct code.',
        'logic':   'You are a logical reasoning expert. Choose the best answer.',
        'science': 'You are a science expert. Answer precisely and correctly.',
        'agent':   'You are a planning agent. Break the task into steps and execute.',
    }
    s = SYSTEMS.get(task, SYSTEMS['math'])
    return (f'<|im_start|>system\n{s}\n<|im_end|>\n'
            f'<|im_start|>user\n{question.strip()}\n<|im_end|>\n'
            f'<|im_start|>assistant\n')

def parse_answer(text: str, task: str = 'math') -> str:
    if task in ('math', 'science'):
        m = re.search(r'####\s*([\-0-9,\.]+)', text)
        if m:
            try: return str(int(float(m.group(1).replace(',',''))))
            except: return m.group(1)
    if task in ('logic', 'code', 'science'):
        m = re.match(r'^([A-D])[.):] ?', text.strip())
        if m: return m.group(1)
    nums = re.findall(r'\b-?\d+(?:\.\d+)?\b', text.replace(',',''))
    if nums:
        try: return str(int(float(nums[-1])))
        except: return nums[-1]
    return text.strip()[:20]

@torch.no_grad()
def run_base(prompt: str) -> Tuple[str, float, int]:
    tokenizer = backbone.tokenizer
    tokenizer.padding_side = 'left'
    enc = tokenizer(prompt, return_tensors='pt', max_length=MAX_Q_LEN, truncation=True).to(DEVICE)
    t0  = time.time()
    gid = backbone.llm.generate(
        input_ids=enc['input_ids'], attention_mask=enc['attention_mask'],
        max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id)
    elapsed  = time.time() - t0
    new_toks = gid[0, enc['input_ids'].size(1):]
    return tokenizer.decode(new_toks, skip_special_tokens=True), elapsed, len(new_toks)

@torch.no_grad()
def run_cida(prompt: str) -> Tuple[str, float, int, dict]:
    tokenizer = backbone.tokenizer
    llm       = backbone.llm
    llm_dtype = torch.float16 # Since we use 4bit nf4, input embeds need to be matched to computation dtype
    tokenizer.padding_side = 'left'
    enc = tokenizer(prompt, return_tensors='pt', max_length=MAX_Q_LEN, truncation=True).to(DEVICE)
    pid, pmsk = enc['input_ids'], enc['attention_mask']

    out    = llm(input_ids=pid, attention_mask=pmsk, output_hidden_states=True, use_cache=False)
    last_h = out.hidden_states[-1]
    sidx   = pmsk.sum(1) - 1
    pooled = last_h[0, sidx[0]].float().unsqueeze(0)

    cout     = cida_loader.cida(pooled=pooled)
    plan_emb = cida_loader.proj(cout['state_final'].s)

    embed_layer = llm.get_input_embeddings()
    p_emb = embed_layer(pid).float()
    cat   = torch.cat([p_emb, plan_emb], dim=1).to(llm_dtype) # Match dtype
    P     = NUM_PLAN_TOKENS
    ext_m = torch.cat([pmsk, torch.ones(1,P,device=DEVICE,dtype=pmsk.dtype)], dim=1)

    t0  = time.time()
    gid = llm.generate(
        inputs_embeds=cat, attention_mask=ext_m,
        max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id)
    elapsed  = time.time() - t0
    text     = tokenizer.decode(gid[0], skip_special_tokens=True)
    diag = {'disagreement': cout['disagreement'].item(), 'rounds': cout['rounds_used'], 'p_final': cout['p_final'][0].tolist()}
    return text, elapsed, len(gid[0]), diag

print('Inference engine ready.')


## 4 · Multi-Domain Benchmark Loader

In [ ]:
class BenchmarkSuite:
    def __init__(self, n: int = EVAL_N_FAST):
        self.n = n
        self.benchmarks: Dict[str, List[dict]] = {}
        self._load_all()

    def _load_all(self):
        print('Loading benchmarks...')
        self.benchmarks['math']    = self._gsm8k()
        self.benchmarks['logic']   = self._arc()
        self.benchmarks['science'] = self._sciqa()
        print('Benchmarks loaded:')
        for k, v in self.benchmarks.items():
            print(f'  {k:<12} {len(v):>4} samples')

    def _gsm8k(self):
        raw = list(load_dataset('openai/gsm8k','main',split='test'))
        random.shuffle(raw)
        out = []
        for ex in raw[:self.n]:
            m = re.search(r'####\s*([\-0-9,\.]+)', ex['answer'])
            if m:
                g = str(int(float(m.group(1).replace(',',''))))
                out.append({'question': ex['question'], 'gold': g, 'task': 'math', 'source': 'gsm8k'})
        return out

    def _arc(self):
        raw = list(load_dataset('allenai/ai2_arc','ARC-Challenge',split='test'))
        random.shuffle(raw)
        out = []
        for ex in raw[:self.n]:
            ch  = ex['choices']
            ak  = ex['answerKey'].strip().upper()
            ct  = '\n'.join(f'{l}. {t}' for l,t in zip(ch['label'],ch['text']))
            out.append({'question': f"{ex['question']}\n{ct}", 'gold': ak, 'task': 'logic', 'source': 'arc_challenge'})
        return out

    def _sciqa(self):
        try:
            raw = list(load_dataset('sciq', split='test'))
        except Exception:
            return []
        random.shuffle(raw)
        out = []
        for ex in raw[:self.n]:
            choices = [ex.get('correct_answer',''), ex.get('distractor1',''), ex.get('distractor2',''), ex.get('distractor3','')]
            random.shuffle(choices)
            correct_idx = choices.index(ex.get('correct_answer',''))
            labels = ['A','B','C','D']
            ct = '\n'.join(f'{labels[i]}. {c}' for i,c in enumerate(choices))
            out.append({'question': f"{ex['question']}\n{ct}", 'gold': labels[correct_idx], 'task': 'science', 'source': 'sciq'})
        return out

suite = BenchmarkSuite(n=EVAL_N_FAST)


## 5 · Core Evaluation Function

In [ ]:
def is_correct(pred: str, gold: str, task: str) -> bool:
    p = parse_answer(pred, task)
    g = str(gold).strip().upper()
    p = str(p).strip().upper()
    return p == g or pred.strip().upper().startswith(g)

def evaluate_config(domain: str, use_cida: bool, n: int = EVAL_N_FAST, verbose: bool = False) -> dict:
    samples = suite.benchmarks.get(domain, [])[:n]
    if not samples: return {'acc': 0, 'n': 0, 'error': 'no samples'}
    correct, times, token_counts, disagreements = 0, [], [], []
    label = f"{'CIDA' if use_cida else 'BASE'}/{domain}"
    for ex in tqdm(samples, desc=label, leave=False):
        prompt = make_prompt(ex['question'], ex['task'])
        try:
            if use_cida:
                text, elapsed, ntok, diag = run_cida(prompt)
                disagreements.append(diag['disagreement'])
            else:
                text, elapsed, ntok = run_base(prompt)
            times.append(elapsed)
            token_counts.append(ntok)
            if is_correct(text, ex['gold'], ex['task']): correct += 1
        except Exception as e:
            continue
    n_valid = len(times)
    return {'acc': correct / max(n_valid, 1), 'n': n_valid, 'avg_time_s': sum(times)/max(n_valid, 1), 'avg_tokens': sum(token_counts)/max(n_valid,1)}


## 6 · Results Store

In [ ]:
results = defaultdict(lambda: defaultdict(dict))
def record(backbone_label, cida_label, domain, metrics):
    results[backbone_label][cida_label][domain] = metrics
    acc = metrics.get('acc', 0)
    print(f'  {backbone_label:<14} {cida_label:<18} {domain:<10}  acc={acc*100:5.1f}%')


## 7 · Main Experiment

In [ ]:
DOMAINS = ['math', 'logic', 'science']
ACTIVE_BACKBONES = ['qwen2.5-1.5b'] # Just run 1 for now to test stability
print('KAIROS-CIDA SCIENTIFIC VALIDATION')
for bb_label in ACTIVE_BACKBONES:
    bb_name = LLM_BACKBONES[bb_label]
    backbone.load(bb_name, bb_label)
    print(f'  --- BASE LLM (no CIDA) ---')
    for domain in DOMAINS:
        record(bb_label, 'base_no_cida', domain, evaluate_config(domain, use_cida=False))
    for cida_label, cida_repo in CIDA_VERSIONS.items():
        try:
            cida_loader.load(cida_repo, cida_label, backbone.hidden_size)
        except Exception as e:
            print(f'  Skipping {cida_label}: {e}')
            continue
        for domain in DOMAINS:
            record(bb_label, cida_label, domain, evaluate_config(domain, use_cida=True))
        cida_loader.unload()
    backbone.unload()
print('\nAll experiments complete.')
